# PagedAttention & KV-Cache Management

Wiki reference for [PagedAttention & KV-Cache Management](https://ml-viz-ruby.vercel.app/wiki/paged-attention).

> **Tip:** use *File → Save a copy in Drive* so your edits persist.

**The idea in one sentence.** Instead of reserving one contiguous `max_seq_len` slab of KV-cache
per request (60–80% of which is never used), carve memory into small fixed-size **blocks**, map
each sequence's tokens to blocks through a **block table**, share identical prefix blocks between
requests via **reference counts**, and copy-on-write when a shared block must diverge — exactly
how an OS pages RAM. We build the allocator from scratch, reproduce the wiki's worked trace, and
measure the waste it eliminates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(11)

## From scratch: a block manager with prefix sharing

Three pieces, mirroring vLLM's design (simplified — real block hashes include model/config salt,
and evicted blocks linger in an LRU cache):

- a **free list** of physical block ids and a **refcount** per allocated block;
- a **prefix-hash table**: every *full* block is keyed by the entire token prefix up to and
  including that block, so a new request whose prompt matches an existing chain can point at the
  same physical blocks (refcount++) instead of re-prefilling them;
- per-sequence **block tables** mapping logical block index → physical block id. Token $t$ of a
  sequence lives at physical slot $\text{table}[\lfloor t/B \rfloor] \cdot B + (t \bmod B)$.

In [ ]:
class BlockManager:
    def __init__(self, num_blocks, block_size):
        self.B = block_size
        self.free_list = list(range(num_blocks))
        self.refcount = {}
        self.prefix_to_block = {}          # hash(prefix incl. this block) -> physical id

    def alloc(self):
        if not self.free_list:
            raise MemoryError('out of KV blocks -> scheduler must preempt someone')
        b = self.free_list.pop()
        self.refcount[b] = 1
        return b

class Sequence:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.table = []                    # logical block idx -> physical block id
        self.prefill_work = 0              # tokens actually prefilled (not cache-hit)

def allocate_prompt(mgr, tokens):
    """Admit a prompt: share full prefix blocks on hash hit, allocate the rest."""
    seq = Sequence(tokens)
    n_full = len(tokens) // mgr.B
    for i in range(n_full):
        prefix = tuple(tokens[: (i + 1) * mgr.B])
        h = hash(prefix)
        if h in mgr.prefix_to_block:                   # prefix cache HIT
            b = mgr.prefix_to_block[h]
            mgr.refcount[b] += 1
        else:                                          # MISS -> allocate + prefill
            b = mgr.alloc()
            mgr.prefix_to_block[h] = b
            seq.prefill_work += mgr.B
        seq.table.append(b)
    tail = len(tokens) - n_full * mgr.B                # partial tail block: never shared
    if tail:
        seq.table.append(mgr.alloc())
        seq.prefill_work += tail
    return seq

def append_token(mgr, seq, tok):
    """One decode step: only allocate when the last block is exactly full."""
    if len(seq.tokens) % mgr.B == 0:
        seq.table.append(mgr.alloc())
    seq.tokens.append(tok)

def free_sequence(mgr, seq):
    """Return blocks whose refcount hits zero to the free list."""
    for b in seq.table:
        mgr.refcount[b] -= 1
        if mgr.refcount[b] == 0:
            mgr.free_list.append(b)
            mgr.prefix_to_block = {h: pb for h, pb in mgr.prefix_to_block.items() if pb != b}
    seq.table = []

mgr = BlockManager(num_blocks=8, block_size=4)
print(f'manager ready: {len(mgr.free_list)} blocks of {mgr.B} tokens')

## Reproduce the worked trace

Block size $B = 4$. Request **A**: a 10-token prompt (2 full blocks + a 2-token tail).
Request **B**: shares A's first 8 tokens, then 3 of its own — its block table should *point at
A's first two physical blocks* (refcount 2) and prefill only 3 tokens instead of 11.

In [ ]:
A = allocate_prompt(mgr, list(range(10)))            # tokens t0..t9
print(f'A table: {A.table}, prefill work: {A.prefill_work}')

B_req = allocate_prompt(mgr, list(range(8)) + [100, 101, 102])
print(f'B table: {B_req.table}, prefill work: {B_req.prefill_work}')

assert B_req.table[:2] == A.table[:2], 'B shares A\'s two full prefix blocks'
assert mgr.refcount[A.table[0]] == 2 and mgr.refcount[A.table[1]] == 2
assert B_req.prefill_work == 3, 'B prefills 3 tokens instead of 11 (73% saved)'

# A generates: t10, t11 fill its tail block; the 13th token pops a fresh block
append_token(mgr, A, 10); append_token(mgr, A, 11)
blocks_before = len(A.table)
append_token(mgr, A, 12)
assert len(A.table) == blocks_before + 1, 'a new block is allocated only when the tail fills'

# A finishes -> its private blocks return; the shared ones stay alive for B
free_before = len(mgr.free_list)
free_sequence(mgr, A)
assert len(mgr.free_list) == free_before + 2, 'A frees its 2 private blocks'
assert mgr.refcount[B_req.table[0]] == 1, 'shared blocks survive with refcount 1'
print('worked trace reproduced: sharing, refcounts, tail growth, and freeing all check out')

## Measure the waste: contiguous reservation vs paging

A contiguous allocator reserves `max_seq_len` slots per request up front. The paged allocator's
only waste is the unfilled tail of each sequence's last block — at most $B-1$ slots. We simulate
200 finished requests (final lengths 50–500 tokens, `max_seq_len = 512`) and compare.

In [ ]:
max_seq_len, B_size = 512, 16
lengths = rng.integers(50, 501, 200)

naive_reserved = len(lengths) * max_seq_len
used = lengths.sum()
naive_waste = 1 - used / naive_reserved

paged_allocated = (np.ceil(lengths / B_size) * B_size).sum()
paged_waste = 1 - used / paged_allocated

print(f'tokens actually cached:    {used:8d}')
print(f'contiguous reserved:       {naive_reserved:8d}  -> {naive_waste:6.1%} wasted')
print(f'paged allocated (B=16):    {int(paged_allocated):8d}  -> {paged_waste:6.1%} wasted')

assert naive_waste > 0.40, 'contiguous allocation wastes most of its reservation'
assert paged_waste < 0.05, 'paged waste is bounded by the last-block slack'

# the waste IS the concurrency: sequences that fit in a fixed pool of HBM
pool = 64 * max_seq_len                                # e.g. room for 64 naive reservations
fit_naive = pool // max_seq_len
fit_paged = int(pool // (np.ceil(lengths.mean() / B_size) * B_size))
print(f'\nfixed pool fits {fit_naive} sequences (contiguous) vs ~{fit_paged} (paged) '
      f'-> {fit_paged / fit_naive:.1f}x more batch slots')

## Visualize: where the memory goes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = ['contiguous\n(max_seq_len)', 'paged\n(B = 16)']
used_bar = [used, used]
waste_bar = [naive_reserved - used, paged_allocated - used]
ax.bar(labels, used_bar, color='#4ade80', label='tokens cached (useful)')
ax.bar(labels, waste_bar, bottom=used_bar, color='#f87171', label='reserved but unused')
ax.set_ylabel('KV-cache slots')
ax.set_title('The same 200 requests under both allocators')
ax.legend()
plt.tight_layout(); plt.show()

**What to notice.**

- The green (useful) bars are identical — paging changes nothing about what must be stored,
  only what must be *reserved*. The red waste is what naive allocation burns on requests that
  might have grown to `max_seq_len` but didn't.
- Reclaimed waste converts directly into **batch slots**, which is why PagedAttention's
  headline win is *throughput* (2–4×), not latency: more sequences share each weight-read.
- Shrink the block size and the residual red sliver shrinks too — but block tables grow and
  the attention kernel's gather overhead rises. $B = 16$ is the empirical sweet spot.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **volatile tokens early in the prompt** | a timestamp or user-id in the system prompt breaks the hash chain for everything after it — put volatile content last |
| **partial tail blocks never share** | prefix reuse works in whole-block units; short shared prefixes (< B tokens) get no hit |
| **refcount leaks** | a sequence freed twice (or never) corrupts the pool — the asserts above are the real invariants vLLM tests |
| **preemption storms** | when the free list empties, victims are swapped/recomputed; throughput silently becomes recompute work — watch the preemption counter |
| **block size too small** | bigger tables + gather overhead; too big → tail waste and worse sharing granularity |

## ✏️ Your turn

**Exercise.** Implement `cow_write(mgr, seq, logical_idx)` — **copy-on-write**: a sequence is
about to *modify* logical block `logical_idx` (e.g. parallel-sampling branches diverging).
If the physical block is private (refcount 1), writing in place is safe — return it. If it's
shared, release our reference, allocate a fresh private block, update the table, and return
the new block.

In [ ]:
def cow_write(mgr, seq, logical_idx):
    """Return a physical block this sequence may safely write to."""
    b = seq.table[logical_idx]
    if mgr.refcount[b] == 1:
        return b                       # private -> write in place
    # TODO(you): decrement the shared block's refcount
    # TODO(you): allocate a fresh block, point seq.table[logical_idx] at it
    # TODO(you): return the new physical block
    ...

mgr2 = BlockManager(num_blocks=8, block_size=4)
s1 = allocate_prompt(mgr2, list(range(8)))
s2 = allocate_prompt(mgr2, list(range(8)))     # shares both blocks with s1
print('s1:', s1.table, ' s2:', s2.table)

In [ ]:
# Assertion — passes silently when your implementation is correct
shared = s2.table[0]
assert mgr2.refcount[shared] == 2, 'setup: block is shared'
new_block = cow_write(mgr2, s2, 0)
assert new_block != shared, 'shared block must be copied, not written in place'
assert s2.table[0] == new_block and s1.table[0] == shared, 'only s2 repoints'
assert mgr2.refcount[shared] == 1 and mgr2.refcount[new_block] == 1
assert cow_write(mgr2, s2, 0) == new_block, 'private block: write in place'
print('all checks passed — copy-on-write works')

<details><summary>Solution</summary>

```python
def cow_write(mgr, seq, logical_idx):
    b = seq.table[logical_idx]
    if mgr.refcount[b] == 1:
        return b
    mgr.refcount[b] -= 1
    new_b = mgr.alloc()
    seq.table[logical_idx] = new_b
    return new_b
```

(A real implementation also copies the K/V tensor contents from `b` to `new_b` — here we only
track the bookkeeping.)
</details>

## Key takeaways

- **Block tables turn the KV-cache into paged memory**: allocate $B$-token blocks on demand;
  waste collapses from 60–80% to under $B{-}1$ slots per sequence (verified: ~44% → ~2%).
- **Prefix sharing is refcounting on content-hashed full blocks** — a matching prompt prefix
  costs zero prefill (verified: 3 tokens instead of 11), which is exactly the prompt-caching
  discount API providers pass on.
- **Copy-on-write** lets shared blocks diverge safely — the same trick `fork()` uses.
- Reclaimed memory = **more concurrent sequences per weight-read** = the 2–4× throughput win.

**Next:** [the wiki page](https://ml-viz-ruby.vercel.app/wiki/paged-attention) ·
[Continuous Batching](https://ml-viz-ruby.vercel.app/wiki/continuous-batching) ·
[Optimizing LLM Inference](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/22-optimizing-llm-inference)